### Image Generation  using pre trained model

In [ ]:
# Install Required Libraries
!pip install pytorch-pretrained-biggan transformers nltk

In [ ]:
import torch
from pytorch_pretrained_biggan import (
    BigGAN,
    one_hot_from_names,
    truncated_noise_sample
)

import nltk
from PIL import Image
from IPython.display import display

In [ ]:
# Download category dataset
nltk.download('wordnet')

# Load Pretrained BigGAN Model
model = BigGAN.from_pretrained('biggan-deep-256')

In [ ]:
# Define Category
category = ['soap bubble']

# Truncation Value
truncation = 0.4

# Create Class Vector
class_vector = one_hot_from_names(category, batch_size=1)

# Create Noise Vector
noise_vector = truncated_noise_sample(
    batch_size=1,
    truncation=truncation
)

In [ ]:
# Convert to Tensor
noise_vector = torch.from_numpy(noise_vector)
class_vector = torch.from_numpy(class_vector)

In [ ]:
# Generate Image
with torch.no_grad():
    output = model(
        noise_vector,
        class_vector,
        truncation
    )

In [ ]:
# Convert Tensor to Image
output = output.cpu().squeeze().numpy()

output = ((output + 1.0) / 2.0) * 255

img = Image.fromarray(
    output.astype('uint8').transpose(1, 2, 0)
)

In [ ]:
# Save and Display
img.save("generated.png")

display(img)

### Text Generation using Pre trained model

In [ ]:
from transformers import pipeline, set_seed

In [ ]:
# Load GPT-2 Model
generator = pipeline(
    'text-generation',
    model='gpt2'
)

In [ ]:
# Random Seed
set_seed(42)

# Prompt
prompt = "The secret door in the library leads to"

In [ ]:
# Generate Text
results = generator(
    prompt,
    max_length=50,
    num_return_sequences=1,
    truncation=True
)

In [ ]:
# Print Result
print(results[0]['generated_text'])

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

import torch

In [ ]:
# Load DialoGPT Model
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/DialoGPT-medium"
)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/DialoGPT-medium"
)

In [ ]:
# User Input
user_input = "what os is safest "

In [ ]:
# Encode Input
new_user_input_ids = tokenizer.encode(
    user_input + tokenizer.eos_token,
    return_tensors='pt'
)

In [ ]:
# Generate Response
chat_history_ids = model.generate(
    new_user_input_ids,
    max_length=1000,
    pad_token_id=tokenizer.eos_token_id,
    no_repeat_ngram_size=3,
    do_sample=True,
    top_k=100,
    top_p=0.7,
    temperature=0.8
)

In [ ]:
# Decode Response
response = tokenizer.decode(
    chat_history_ids[:, new_user_input_ids.shape[-1]:][0],
    skip_special_tokens=True
)

In [ ]:
# Print Chat
print("User:", user_input)
print("AI:", response)